# Query DSL — operator tour

Every operator in `meta/07_query_language.md` demonstrated against
the real corpus. Each cell prints the clip count for that query.

The DSL grammar:

* `<entity>.<attr> <cmp> <value>` — basic predicate.
* `<entity>(<expr>)` — same-entity grouping.
* Boolean: `and`, `or`, `not`.
* Temporal: `while`, `then(K)`.
* Relational: `because_of`.
* Scoping: `within W: E`.


In [ ]:
from pathlib import Path
from causal_ai_av.dataset import CausalAVDataset

ds = CausalAVDataset(Path("/home/horde/01_json_annotations"))

def show(q: str) -> None:
    print(f"{ds.count(q):>4}  {q}")


## Attribute predicates


In [ ]:
show("agent.type = ped")
show("ego.action = decel")
show("light.color = red")
show("env.lanes >= 2")


## Hierarchical aliases

`vehicle`, `vru`, `intersection`, `turn`, `change_lane` are *parent*
aliases — they expand to the union of their children.


In [ ]:
show("agent.type = vehicle")
show("agent.type = vru")
show("agent.type = cyclist")
show("env.type = intersection")


## Set membership — `in (a, b, c)` is OR


In [ ]:
show("ego.action in (stop, yield, decel)")
show("agent.type in (ped, cyclist, animal)")


## Same-entity coupling — `agent(...)`

Inside an entity clause, the constraints apply to the **same** agent.
Compare to free-floating predicates which may match different agents.


In [ ]:
show("agent(type = vehicle, pos = front)")
show("agent.type = vehicle and agent.pos = front")


## Boolean operators


In [ ]:
show("agent.type = ped and env.type = crosswalk")
show("ego.action = stop or ego.action = yield")
show("agent.type = ped and not env.type = crosswalk")


## Action flags

Action types in the corpus encode flags as parenthesized suffixes
(`oxd:Walk (jaywalk)`). Flag attributes match either the schema
flag field OR the suffix token.


In [ ]:
show("agent(type = ped, action(jaywalk = true))")
show("agent(action(erratic = true))")


## Temporal — `while`

Pairs of matches whose intervals intersect.


In [ ]:
show("agent.type = ped while ego.action = decel")
show("light.color = red while ego.action = stop")


## Temporal — `then(K)`

B starts during A or within K seconds after A ends. Default K=0
means must touch or overlap.


In [ ]:
show("light.color = yellow then(3) ego.action = stop")
show("light.color = green then ego.action = drive")


## Relational — `because_of`

Follows the schema's `because_of` causal edge. Both halves are
emitted as match tuples.


In [ ]:
show("ego.action = decel because_of agent.type = ped")
show("ego.action = drive because_of agent.type = ped")


## Scoping — `within W: E`

Restricts E's temporal window to W's intervals. Inside, `not E`
means "E doesn't happen during W."


In [ ]:
show("within light.color = red: not ego.action = stop")
show("within env.type = crosswalk: agent.type = ped")


## Annotator-tagged flags


In [ ]:
show("light(color = yellow, ego_in_on_yellow = true)")
show("light(color = yellow, could_have_cleared = true)")
